# ModernBERT by hand

**Part 2 of "LLMs the Hard Way"** (branch `modernbert-by-hand`). The decoder notebooks on `master` built GPT-2 and SmolLM2 by hand, ending in a KV-cache decode step. This notebook does the same for an **encoder**: [answerdotai/ModernBERT-large](https://huggingface.co/answerdotai/ModernBERT-large), loaded from its real checkpoint as a plain dict of tensors, with no library model.

We implement and inspect every piece ourselves, then check the result against the Hugging Face implementation:

- tokenization and token embeddings;
- attention, briefly (Q, K, V, heads and RoPE are built step by step in the decoder notebooks and `rope.ipynb` on `master`), with the focus on what an encoder changes: **bidirectional** attention (no causal mask), compared with SmolLM2's causal attention;
- local (sliding-window) vs global attention layers, each with its own RoPE base;
- LayerNorm, residual connections and the GeGLU MLP;
- one complete encoder layer, then the full 28-layer stack;
- the output `[batch, sequence, hidden]`: token representations and mean pooling;
- **a classification head on top of the encoder**, trained and tested, and what it costs to add a class.

**The end result:** our forward pass matches the reference `last_hidden_state` to about 3e-5, and the checkpoint's own masked-LM head fills in masked words correctly.

**The idea this notebook ends on.** An encoder turns text into vectors; a small **head** on the pooled vector turns it into a model that decides (a class, a yes / no, a score). Phase 8 trains such a head, and shows what it costs to add a class: that cost is why `02_frozen_router.ipynb` (branch `frozen-router`) changes the head.

**Runs on:** a CPU is enough (the comparisons with the reference model run on CPU on purpose, for exact numbers). On Kaggle or Colab, run it top to bottom.

In [ ]:
# Cell 1 — Environment sanity check

import sys
import torch

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device :", "CUDA" if torch.cuda.is_available() else "CPU")
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

In [ ]:
!pip install -q "transformers==5.17.0"

---
# Phase 1 — Encoder vs decoder

## Cell 2 — Who can see whom

A decoder (SmolLM2) lets token `i` attend only to tokens `≤ i` (causal mask), because it predicts the next token. An encoder (ModernBERT) lets every token attend to every token, because it builds each token's meaning from both sides; it's pretrained by recovering masked words (Cell 15). Cell 7 shows the same difference on real weights.

In [ ]:
# Cell 2 — Visibility: which keys each query may attend to

tokens = ["the", "cat", "sat", "down"]
S = len(tokens)

# Decoder (SmolLM2): query i may attend to keys 0..i  -> lower-triangular
decoder_visibility = torch.tril(torch.ones(S, S, dtype=torch.bool))

# Encoder (ModernBERT): query i may attend to every key -> all True
encoder_visibility = torch.ones(S, S, dtype=torch.bool)

print("decoder_visibility:", tuple(decoder_visibility.shape), "| encoder_visibility:", tuple(encoder_visibility.shape))
print()


def show_visibility(title, visibility):
    print(title)
    print("query \\ key".ljust(13) + "".join(t.ljust(6) for t in tokens))
    for i, t in enumerate(tokens):
        row = "".join(("✓" if visibility[i, j] else "·").ljust(6) for j in range(S))
        print(t.ljust(13) + row)
    print()


show_visibility("Decoder (causal) — SmolLM2", decoder_visibility)
show_visibility("Encoder (bidirectional) — ModernBERT", encoder_visibility)

assert decoder_visibility.sum().item() == S * (S + 1) // 2
assert encoder_visibility.sum().item() == S * S

## Cell 3 — Configuration and weights, from the real checkpoint

We read `config.json` and `model.safetensors` ourselves: no library model is built, only a dict of tensors.

| Field | Meaning | ModernBERT-large (used) | ModernBERT-base |
|---|---|---|---|
| `vocab_size` (V) | rows in the embedding table | 50,368 | 50,368 |
| `hidden_size` (H) | width of every token vector | 1,024 | 768 |
| `num_layers` (L) | encoder blocks | 28 | 22 |
| `num_attention_heads` | heads × `head_dim` | 16 × 64 | 12 × 64 |
| `intermediate_size` (I) | MLP inner width | 2,624 | 1,152 |
| `max_position_embeddings` | longest sequence | 8,192 | 8,192 |
| `local_attention_window` | window of local layers | 128 (±64) | 128 (±64) |
| `global_attn_every_n_layers` | every n-th layer is global | 3 | 3 |
| `global_rope_theta` / `local_rope_theta` | RoPE bases | 160,000 / 10,000 | 160,000 / 10,000 |

Set `CHECKPOINT` to switch models; every cell reads its sizes from `config`. The weights take about 1.6 GB in float32 (plus ~1.6 GB for the reference model loaded in Cell 5).

In [ ]:
# Cell 3a — Download the checkpoint's config and weights

import json
from dataclasses import dataclass

from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

torch.manual_seed(42)

# Which pretrained checkpoint to load. Both share the architecture built in this notebook:
#   "answerdotai/ModernBERT-base"  : 22 layers, H = 768,  12 heads, I = 1152  (149M parameters)
#   "answerdotai/ModernBERT-large" : 28 layers, H = 1024, 16 heads, I = 2624  (395M parameters)
CHECKPOINT = "answerdotai/ModernBERT-large"

config_path = hf_hub_download(CHECKPOINT, "config.json")
weights_path = hf_hub_download(CHECKPOINT, "model.safetensors")

with open(config_path) as f:
    raw_config = json.load(f)

### Cell 3b — The configuration as our own object

The fields the forward pass needs, plus `head_dim` and which layers are global.

In [ ]:
# Cell 3b — The configuration as our own object

@dataclass
class ModernBertConfig:
    vocab_size: int
    hidden_size: int
    num_layers: int
    num_attention_heads: int
    intermediate_size: int
    max_position_embeddings: int
    local_attention_window: int
    global_attn_every_n_layers: int
    global_rope_theta: float
    local_rope_theta: float
    norm_eps: float
    pad_token_id: int
    cls_token_id: int
    sep_token_id: int

    @property
    def head_dim(self):
        return self.hidden_size // self.num_attention_heads

    def is_global_layer(self, layer_idx):
        return layer_idx % self.global_attn_every_n_layers == 0


config = ModernBertConfig(
    vocab_size=raw_config["vocab_size"],
    hidden_size=raw_config["hidden_size"],
    num_layers=raw_config["num_hidden_layers"],
    num_attention_heads=raw_config["num_attention_heads"],
    intermediate_size=raw_config["intermediate_size"],
    max_position_embeddings=raw_config["max_position_embeddings"],
    local_attention_window=raw_config["local_attention"],
    global_attn_every_n_layers=raw_config["global_attn_every_n_layers"],
    global_rope_theta=raw_config["global_rope_theta"],
    local_rope_theta=raw_config["local_rope_theta"],
    norm_eps=raw_config["norm_eps"],
    pad_token_id=raw_config["pad_token_id"],
    cls_token_id=raw_config["cls_token_id"],
    sep_token_id=raw_config["sep_token_id"],
)

### Cell 3c — Inspect it

In [ ]:
# Cell 3c — Inspect the configuration

assert config.hidden_size % config.num_attention_heads == 0, "H must split evenly across heads"
assert config.head_dim % 2 == 0, "RoPE rotates pairs of dimensions"

print(f"Checkpoint: {CHECKPOINT}\n")
for field in ["vocab_size", "hidden_size", "num_layers", "num_attention_heads", "head_dim",
              "intermediate_size", "max_position_embeddings", "local_attention_window",
              "global_attn_every_n_layers", "global_rope_theta", "local_rope_theta", "norm_eps"]:
    print(f"  {field:28s} {getattr(config, field)}")

### Cell 3d — The weights as a plain dict

The masked-LM head is set aside here; Cell 15 uses it once.

In [ ]:
# Cell 3d — The pretrained weights as a plain dict of tensors

# ---- Pretrained weights as a plain dict of tensors (no library model) ----
checkpoint_tensors = load_file(weights_path)
weights = {name.removeprefix("model."): t for name, t in checkpoint_tensors.items() if name.startswith("model.")}
mlm_head = sorted(name for name in checkpoint_tensors if not name.startswith("model."))
del checkpoint_tensors

for t in weights.values():
    t.requires_grad_(False)

print(f"\nLoaded {len(weights)} encoder tensors; set aside the masked-LM head: {mlm_head}\n")
for name, t in weights.items():
    if ".layers." not in "." + name or name.startswith(("layers.0.", "layers.1.")):
        print(f"  {name:36s} {str(tuple(t.shape)):14s} {t.dtype}")
print(f"  ... layers.2 – layers.{config.num_layers - 1} repeat the layers.1 pattern")

### Cell 3e — Count the parameters

There are no biases anywhere, so the count follows from the shapes: `V·H + H` (embeddings + norm), then per layer `3H·H` (`Wqkv`) + `H·H` (`Wo`) + `H·2I` (`Wi`) + `I·H` (MLP `Wo`) + `2H` (norms), minus `H` for layer 0 (no attention norm), plus `H` (final norm). **Expect: 394,781,696.**

In [ ]:
# Cell 3e — Check the parameter count against the formula

H, I, V, L = config.hidden_size, config.intermediate_size, config.vocab_size, config.num_layers
expected = (V * H + H) + L * (3 * H * H + H * H + H * 2 * I + I * H + 2 * H) - H + H
actual = sum(t.numel() for t in weights.values())
print(f"\nEncoder parameters: {actual:,} (formula: {expected:,})")

assert actual == expected
assert "layers.0.attn_norm.weight" not in weights, "layer 0 has no attention norm"
assert weights["layers.0.attn.Wqkv.weight"].shape == (3 * H, H)
assert weights["layers.0.mlp.Wi.weight"].shape == (2 * I, H)

---
# Phase 2 — Input

## Cell 4 — Token IDs

ModernBERT's own byte-level BPE tokenizer (50,368 entries); `Ġ` marks a leading space. Special tokens: `[CLS]` 50281 (first), `[SEP]` 50282 (end of a segment), `[PAD]` 50283 (padding, mask 0), `[MASK]` 50284, `[UNK]` 50280.

In [ ]:
# Cell 4a — The tokenizer and its special tokens

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

special = ["[CLS]", "[SEP]", "[PAD]", "[MASK]", "[UNK]"]
special_ids = {t: tokenizer.convert_tokens_to_ids(t) for t in special}
print("Vocabulary size:", len(tokenizer))
print("Special token IDs:", special_ids)

assert len(tokenizer) == config.vocab_size
assert special_ids["[CLS]"] == config.cls_token_id
assert special_ids["[SEP]"] == config.sep_token_id
assert special_ids["[PAD]"] == config.pad_token_id

CLS_ID, SEP_ID, PAD_ID, MASK_ID = (special_ids[t] for t in ["[CLS]", "[SEP]", "[PAD]", "[MASK]"])

### Cell 4b — Text → tokens → IDs

In [ ]:
# Cell 4b — Text -> tokens -> token IDs

texts = ["The dog bites the man.", "My card payment failed yesterday, please help!"]

for text in texts:
    ids = tokenizer(text)["input_ids"]
    print(f"\ntext   : {text!r}")
    print(f"tokens : {tokenizer.convert_ids_to_tokens(ids)}")
    print(f"ids    : {ids}")

### Cell 4c — A padded batch and its attention mask

The mask is 1 for real tokens and 0 for `[PAD]`.

In [ ]:
# Cell 4c — A padded batch and its attention mask

batch = tokenizer(texts, padding=True, return_tensors="pt")
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]

print("\ninput_ids:", tuple(input_ids.shape), "= [B, S]")
print(input_ids)
print("attention_mask:", tuple(attention_mask.shape))
print(attention_mask)

assert input_ids.shape == attention_mask.shape
assert (input_ids[:, 0] == CLS_ID).all(), "every sequence starts with [CLS]"
assert ((input_ids == PAD_ID) == (attention_mask == 0)).all(), "mask is 0 exactly where [PAD] is"
for b in range(input_ids.shape[0]):
    last_real = attention_mask[b].sum().item() - 1
    assert input_ids[b, last_real] == SEP_ID, "every sequence ends with [SEP] before padding"

## Cell 5 — Token embeddings

A row lookup, `X = W_emb[input_ids]`: `[B, S]` → `[B, S, 1024]`. ModernBERT has **no position-embedding table** (position comes from RoPE inside attention), and its `[PAD]` row isn't zero (the mask handles padding). The reference model, loaded here for comparison only, must give exactly the same vectors.

In [ ]:
# Cell 5 — Token embeddings: [B, S] -> [B, S, H] using the pretrained table

W_emb = weights["embeddings.tok_embeddings.weight"]      # [V, H]

X = W_emb[input_ids]                                     # our lookup: plain row indexing

B, S = input_ids.shape
H = config.hidden_size

print("embedding table W_emb:", tuple(W_emb.shape), "= [V, H]")
print("input_ids            :", tuple(input_ids.shape), "= [B, S]")
print("X                    :", tuple(X.shape), "= [B, S, H]")

assert X.shape == (B, S, H)
assert torch.equal(X[0, 0], W_emb[CLS_ID]), "[CLS] vector = row CLS_ID of the table"

print("\nFirst 6 dims of each token in sequence 0:")
for s in range(S):
    token = tokenizer.convert_ids_to_tokens(input_ids[0, s].item())
    print(f"  pos {s:2d} {token:10s} {X[0, s, :6]}")

print(f"\n[PAD] row norm: {W_emb[PAD_ID].norm():.3f}  (not zero: padding is handled by the attention mask)")

# ---- Validation against the reference implementation (comparison only) ----
from transformers import ModernBertModel

reference = ModernBertModel.from_pretrained(CHECKPOINT, attn_implementation="eager").eval()

with torch.no_grad():
    X_ref = reference.embeddings.tok_embeddings(input_ids)

max_diff = (X - X_ref).abs().max().item()
print(f"\nours vs reference token embeddings: max |difference| = {max_diff}")
assert torch.equal(X, X_ref)

---
# Phase 3 — Attention in an encoder

Q / K / V, heads, scores, softmax, the weighted sum of V and `Wo` work exactly as in a decoder. They're built step by step on `master` in [`kv_cache_experiment.ipynb`](https://github.com/skamalj/llms-the-hard-way/blob/master/kv_cache_experiment.ipynb) and [`smollm2_forward_pass_kv_cache.ipynb`](https://github.com/skamalj/llms-the-hard-way/blob/master/smollm2_forward_pass_kv_cache.ipynb) (Steps 3–12), and RoPE in [`rope.ipynb`](https://github.com/skamalj/llms-the-hard-way/blob/master/rope.ipynb). Here we run layer 0's attention in three steps, using **ModernBERT's own RoPE**, and check each against the reference. The one big difference, no causal mask, gets Cell 7.

## Cell 6 — Layer-0 attention in three steps

`My card payment failed, so I need a new card.` → `x` `[1, 14, 1024]` → Q, K, V (from one stacked `Wqkv`, no bias) → 16 heads of 64 → scores `[1, 16, 14, 14]` → output `[1, 14, 1024]`.

In [ ]:
# Cell 6a — The layer-0 input, and Q, K, V from the stacked Wqkv

text = "My card payment failed, so I need a new card."
input_ids_1 = tokenizer(text, return_tensors="pt")["input_ids"]          # [1, S]
tokens_1 = tokenizer.convert_ids_to_tokens(input_ids_1[0])
B, S = input_ids_1.shape
H = config.hidden_size
print("tokens:", tokens_1)


def layer_norm(x, weight, eps=config.norm_eps):
    """ModernBERT LayerNorm: normalize each token vector, then scale (no bias; Cell 10)."""
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    return (x - mean) / torch.sqrt(var + eps) * weight


x = layer_norm(W_emb[input_ids_1], weights["embeddings.norm.weight"])   # layer-0 input [B, S, H]
W_q, W_k, W_v = weights["layers.0.attn.Wqkv.weight"].chunk(3, dim=0)   # one stacked [3H, H] matrix -> three [H, H]
Q_proj, K_proj, V_proj = x @ W_q.T, x @ W_k.T, x @ W_v.T                # [B, S, H] each, no bias

print("x:", tuple(x.shape), "| Wqkv:", tuple(weights["layers.0.attn.Wqkv.weight"].shape), "-> Q, K, V:", tuple(Q_proj.shape))
assert Q_proj.shape == K_proj.shape == V_proj.shape == (B, S, H)

### Cell 6b — Heads, RoPE and scores

Layer 0 is a global layer, so RoPE uses the `full_attention` base.

In [ ]:
# Cell 6b — Heads, RoPE (ModernBERT's own implementation) and scores

from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb

n_heads, d = config.num_attention_heads, config.head_dim             # 16 heads of 64


def split_heads(t):
    B_, S_, _ = t.shape
    return t.view(B_, S_, n_heads, d).transpose(1, 2)                  # [B, S, H] -> [B, heads, S, head_dim]


def rope_cos_sin(seq_len, layer_type):
    """RoPE's cos / sin [1, S, head_dim] from ModernBERT's rotary module: "full_attention" for global layers
    (theta = 160,000) or "sliding_attention" for local layers (theta = 10,000)."""
    return reference.rotary_emb(W_emb[None, :1], torch.arange(seq_len)[None], layer_type)


Q_heads, K_heads, V_heads = split_heads(Q_proj), split_heads(K_proj), split_heads(V_proj)
cos_0, sin_0 = rope_cos_sin(S, "full_attention")                       # layer 0 is a global layer
Q_heads_rope, K_heads_rope = apply_rotary_pos_emb(Q_heads, K_heads, cos_0, sin_0)
scores = Q_heads_rope @ K_heads_rope.transpose(-2, -1) / d ** 0.5     # [B, heads, S, S]

print("Q heads:", tuple(Q_heads.shape), "| cos / sin:", tuple(cos_0.shape), "| scores:", tuple(scores.shape))
assert scores.shape == (B, n_heads, S, S)

### Cell 6c — Softmax → weighted values → merge → `Wo`, checked against the reference

In [ ]:
# Cell 6c — Softmax, weighted values, merge, Wo; checked against the reference

attention = torch.softmax(scores, dim=-1)                              # [B, heads, S, S], every key allowed
context = attention @ V_heads                                          # [B, heads, S, head_dim]
merged = context.transpose(1, 2).reshape(B, S, H)                      # [B, S, H]
W_o = weights["layers.0.attn.Wo.weight"]                               # [H, H]
attn_output = merged @ W_o.T                                           # [B, S, H]
print("attention:", tuple(attention.shape), "-> context:", tuple(context.shape), "-> output:", tuple(attn_output.shape))

# The reference model's layer-0 attention module, same input
with torch.no_grad():
    x_ref = reference.embeddings(input_ids_1)                          # embeddings + norm
    cos_ref0, sin_ref0 = reference.rotary_emb(x_ref, torch.arange(S)[None], "full_attention")
    ref_attn_output, ref_attention = reference.layers[0].attn(x_ref, position_embeddings=(cos_ref0, sin_ref0), attention_mask=None)

for name, ours, ref in [("layer-0 input", x, x_ref), ("attention weights", attention, ref_attention),
                        ("attention output", attn_output, ref_attn_output)]:
    diff = (ours - ref).abs().max().item()
    print(f"{name:18s} vs reference: max |diff| = {diff:.1e}")
    assert diff < 1e-4

## Cell 7 — No causal mask

Head 0's real scores under a causal mask (SmolLM2) and with none (ModernBERT). **Expect:** the causal heatmap is lower-triangular; the first `Ġcard` puts a large share of its attention on tokens to its **right**, which a decoder can't do.

In [ ]:
# Cell 7 — Bidirectional (ModernBERT) vs causal (SmolLM2) attention, head 0 of layer 0

import matplotlib.pyplot as plt

causal_mask = torch.tril(torch.ones(S, S, dtype=torch.bool))     # j <= i
bidirectional_mask = torch.ones(S, S, dtype=torch.bool)          # every j

head0 = scores[0, 0]                                             # [S, S]
causal_probs = torch.softmax(head0.masked_fill(~causal_mask, float("-inf")), dim=-1)
bidir_probs = torch.softmax(head0.masked_fill(~bidirectional_mask, float("-inf")), dim=-1)

print("scores (one head):", tuple(head0.shape), "| masks:", tuple(causal_mask.shape))
assert (causal_probs[~causal_mask] == 0).all(), "causal: no attention to the future"
assert (bidir_probs > 0).all(), "bidirectional: every token sees every token"

i = 2   # the first Ġcard
print(f"\nQuery {tokens_1[i]!r} (position {i}) — share of attention on tokens to its RIGHT:")
print(f"  causal       : {causal_probs[i, i + 1:].sum():.3f}")
print(f"  bidirectional: {bidir_probs[i, i + 1:].sum():.3f}")

labels = [t.replace("Ġ", "_") for t in tokens_1]
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, probs, title in [(axes[0], causal_probs, "SmolLM2-style causal: token i → tokens ≤ i"),
                         (axes[1], bidir_probs, "ModernBERT bidirectional: token i → every token")]:
    ax.imshow(probs.numpy(), cmap="Blues", vmin=0, vmax=probs.max().item())
    ax.set_xticks(range(S), labels, rotation=90)
    ax.set_yticks(range(S), labels)
    ax.set_xlabel("key token j")
    ax.set_ylabel("query token i")
    ax.set_title(title)
plt.tight_layout()
plt.show()

---
# Phase 4 — Local and global layers

## Cell 8 — The layer schedule

Every 3rd layer (0, 3, …, 27: 10 layers) is **global**: full attention, RoPE base 160,000. The other 18 are **local**: token `i` sees only tokens with `|i − j| ≤ 64` (still on both sides), RoPE base 10,000. **Expect:** at 8,192 tokens ModernBERT computes about 37% of the attention pairs of an all-global model, and one global layer lets information cross the whole sequence.

In [ ]:
# Cell 8 — Sliding-window mask, layer schedule and cost

def sliding_window_mask(seq_len, half_window, device=None):
    pos = torch.arange(seq_len, device=device)
    return (pos[:, None] - pos[None, :]).abs() <= half_window       # [S, S] bool


w_real = config.local_attention_window // 2            # 64 tokens on each side

print(f"{'layer':>5} | {'attention':>9} | {'range':>12} | {'RoPE theta':>10}")
print("-" * 46)
for i in range(config.num_layers):
    g = config.is_global_layer(i)
    print(f"{i:>5} | {'global' if g else 'local':>9} | {'all tokens' if g else f'±{w_real} tokens':>12} | "
          f"{config.global_rope_theta if g else config.local_rope_theta:>10,.0f}")

n_global = sum(config.is_global_layer(i) for i in range(config.num_layers))
n_local = config.num_layers - n_global
assert n_global == len(range(0, config.num_layers, config.global_attn_every_n_layers))

seq_len = 8192
all_global = config.num_layers * seq_len ** 2
local_pairs = seq_len * (2 * w_real + 1) - w_real * (w_real + 1)
modernbert = n_global * seq_len ** 2 + n_local * local_pairs
print(f"\nAttention pairs for S = {seq_len:,} over {config.num_layers} layers:")
print(f"  all-global : {all_global:,}")
print(f"  ModernBERT : {modernbert:,}  ({modernbert / all_global:.1%} of all-global)")

print("\nHow far can information from token 0 travel (S = 8,192)?")
print(f"  {'after layer':>11} | {'local-only stack':>16} | {'ModernBERT schedule':>19}")
reach_local, reach_mb = 0, 0
for i in range(6):
    reach_local = min(reach_local + w_real, seq_len - 1)
    reach_mb = seq_len - 1 if config.is_global_layer(i) else min(reach_mb + w_real, seq_len - 1)
    print(f"  {i:>11} | {reach_local:>16,} | {reach_mb:>19,}")

## Cell 9 — Masks for a real batch

Padding and the window become one **additive bias** on the scores: 0 where a key is allowed, a huge negative number where it isn't. `self_attention` packages Cell 6 into one function for any layer.

In [ ]:
# Cell 9a — The padding + window bias, and one attention function

import torch.nn.functional as F

NEG_INF = torch.finfo(torch.float32).min   # finite "minus infinity": a row of only blocked keys gives 0, not NaN

def attention_bias(attention_mask, half_window=None):
    """attention_mask [B, S] (1 = real token) -> additive bias [B, 1, S, S] and boolean allowed mask."""
    B_, S_ = attention_mask.shape
    allowed = attention_mask[:, None, None, :].bool().expand(B_, 1, S_, S_)    # keys must be real tokens
    if half_window is not None:
        allowed = allowed & sliding_window_mask(S_, half_window, attention_mask.device)   # and within the window
    bias = torch.zeros(B_, 1, S_, S_, device=attention_mask.device).masked_fill(~allowed, NEG_INF)
    return bias, allowed


def self_attention(x_in, layer_idx, bias, cos, sin):
    """Cells 6a-6c: QKV projection, head split, RoPE, scores, softmax, weighted values, merge, Wo."""
    B_, S_, _ = x_in.shape
    # F.linear(x, W) is x @ W.T, faster on CPU
    q, k, v = F.linear(x_in, weights[f"layers.{layer_idx}.attn.Wqkv.weight"]).chunk(3, dim=-1)  # [B, S, H] each
    Qh, Kh, Vh = split_heads(q), split_heads(k), split_heads(v)                            # [B, heads, S, d]
    Qh, Kh = apply_rotary_pos_emb(Qh, Kh, cos, sin)                                     # ModernBERT's RoPE
    probs = torch.softmax(Qh @ Kh.transpose(-2, -1) / d ** 0.5 + bias, dim=-1)             # [B, heads, S, S]
    out = (probs @ Vh).transpose(1, 2).reshape(B_, S_, H)                                   # [B, S, H]
    return F.linear(out, weights[f"layers.{layer_idx}.attn.Wo.weight"]), probs

### Cell 9b — A toy window (w = 3) on our sentence

The real ±64 window covers a 14-token sentence, so a toy one shows the effect: the second `Ġcard` loses sight of positions 2–4.

In [ ]:
# Cell 9b — A toy window on our 14-token sentence

mask_1 = torch.ones(1, S, dtype=torch.long)
bias_full, allowed_full = attention_bias(mask_1)
bias_toy, allowed_toy = attention_bias(mask_1, half_window=3)

out_full, probs_full = self_attention(x, 0, bias_full, cos_0, sin_0)
out_toy, probs_toy = self_attention(x, 0, bias_toy, cos_0, sin_0)
assert torch.allclose(out_full, attn_output, atol=1e-5), "full-mask path reproduces Cell 6c"

i = 11
print(f"Query {tokens_1[i]!r} (position {i}) may attend to:")
print("  full      :", [tokens_1[j] for j in range(S) if allowed_full[0, 0, i, j]])
print("  local w=3 :", [tokens_1[j] for j in range(S) if allowed_toy[0, 0, i, j]])
print(f"  attention on positions 2-4 — full: {probs_full[0, 0, i, 2:5].sum():.3f} | local: {probs_toy[0, 0, i, 2:5].sum():.3f}")
print(f"  output change for this token: max |diff| = {(out_full[0, i] - out_toy[0, i]).abs().max():.3f}")

### Cell 9c — The real window on a long, padded batch

One sequence over 128 tokens, one short and padded. Our mask must equal the reference's exactly.

In [ ]:
# Cell 9c — A long, padded batch: our mask vs the reference mask

# Validation on a long, padded batch with the real window (w = 64)
from transformers.masking_utils import create_bidirectional_sliding_window_mask

long_texts = [" ".join(["My card payment failed, so I need a new card and a refund for the transfer."] * 9),
              "Short second sequence about investing in a mutual fund."]
long_batch = tokenizer(long_texts, padding=True, return_tensors="pt")
ids_long, mask_long = long_batch["input_ids"], long_batch["attention_mask"]
x_long = layer_norm(W_emb[ids_long], weights["embeddings.norm.weight"])
S_long = ids_long.shape[1]
print(f"\nlong batch: input_ids {tuple(ids_long.shape)}, real tokens per sequence {mask_long.sum(dim=1).tolist()}")

bias_local, allowed_local = attention_bias(mask_long, half_window=w_real)
ref_mask = create_bidirectional_sliding_window_mask(config=reference.config, inputs_embeds=x_long,
                                                    attention_mask=mask_long, allow_is_bidirectional_skip=False)
ref_allowed = ref_mask if ref_mask.dtype == torch.bool else (ref_mask == 0)
assert torch.equal(allowed_local, ref_allowed.expand_as(allowed_local)), "window + padding mask matches reference"
print("our window + padding mask == reference sliding-window mask")

---
# Phase 5 — The encoder layer

## Cell 10 — Residual, LayerNorm and the GeGLU MLP

Residuals, LayerNorm and gated MLPs are built on `master` (GPT-2: LayerNorm, GELU MLP; SmolLM2: SwiGLU). ModernBERT's versions differ only in small ways: **LayerNorm has no bias**, and the MLP is **GeGLU**, SwiGLU with GELU: `Wo · (GELU(a) ⊙ g)`, where `Wi` produces the content `a` and the gate `g` together.

In [ ]:
# Cell 10 — Residual, LayerNorm without a bias, and the GeGLU MLP (layer 0)

h_attn = x + attn_output                                          # residual: h = x + attention(x)

gamma = weights["layers.0.mlp_norm.weight"]                       # [H]: a scale and no bias
mlp_input = layer_norm(h_attn, gamma)                             # normalize each token, then scale

W_i = weights["layers.0.mlp.Wi.weight"]                           # [2I, H]: content and gate, stacked
W_mlp_o = weights["layers.0.mlp.Wo.weight"]                       # [H, I]
a, g = (mlp_input @ W_i.T).chunk(2, dim=-1)                       # [1, S, I] each
mlp_output = (F.gelu(a) * g) @ W_mlp_o.T                          # GeGLU: GELU(content) * gate -> [1, S, H]
print("Wi:", tuple(W_i.shape), "-> a, g:", tuple(a.shape), "-> MLP output:", tuple(mlp_output.shape))

with torch.no_grad():
    ref_mlp_input = reference.layers[0].mlp_norm(h_attn)
    ref_mlp_output = reference.layers[0].mlp(mlp_input)
for name, ours, ref in [("mlp_norm (LayerNorm)", mlp_input, ref_mlp_input), ("MLP (GeGLU)", mlp_output, ref_mlp_output)]:
    diff = (ours - ref).abs().max().item()
    print(f"{name:21s} vs reference: max |diff| = {diff:.1e}")
    assert diff < 1e-4

## Cell 11 — One complete layer

Pre-norm: `h = h + Attn(attn_norm(h))`, then `h = h + MLP(mlp_norm(h))`. Layer 0 has no `attn_norm` (the embedding norm comes right before it). Printed with every shape, and checked against the reference.

In [ ]:
# Cell 11 — One complete pre-norm ModernBERT encoder layer

def mlp(m_in, layer_idx):
    a_, g_ = F.linear(m_in, weights[f"layers.{layer_idx}.mlp.Wi.weight"]).chunk(2, dim=-1)
    return F.linear(F.gelu(a_) * g_, weights[f"layers.{layer_idx}.mlp.Wo.weight"])


def encoder_layer(h, layer_idx, bias, cos, sin, verbose=False):
    show = print if verbose else (lambda *args, **kw: None)
    show(f"layer {layer_idx} ({'global' if config.is_global_layer(layer_idx) else 'local'})")
    show("  input h                 ", tuple(h.shape))
    a_in = h if layer_idx == 0 else layer_norm(h, weights[f"layers.{layer_idx}.attn_norm.weight"])
    show("  attn_norm(h)            ", tuple(a_in.shape), "(identity in layer 0)" if layer_idx == 0 else "")
    attn_out, probs = self_attention(a_in, layer_idx, bias, cos, sin)
    show("  attention probs         ", tuple(probs.shape))
    show("  attention output        ", tuple(attn_out.shape))
    h = h + attn_out
    show("  h + attention           ", tuple(h.shape))
    m_in = layer_norm(h, weights[f"layers.{layer_idx}.mlp_norm.weight"])
    show("  mlp_norm(h)             ", tuple(m_in.shape))
    m_out = mlp(m_in, layer_idx)
    show("  MLP output              ", tuple(m_out.shape))
    h = h + m_out
    show("  h + MLP  (layer output) ", tuple(h.shape))
    return h


h0 = encoder_layer(x, 0, bias_full, cos_0, sin_0, verbose=True)

with torch.no_grad():
    ref_h0 = reference.layers[0](x_ref, attention_mask=None, position_embeddings=(cos_ref0, sin_ref0))
diff = (h0 - ref_h0).abs().max().item()
print(f"\nours vs reference full layer 0: max |diff| = {diff:.1e}")
assert diff < 1e-4

## Cell 12 — A global and a local layer on a padded batch

Layer 0 (global) and layer 1 (local, ±64) on the long padded batch from Cell 9c, compared on every real token.

In [ ]:
# Cell 12 — A padded batch through one global layer (0) and one local layer (1)

bias_global_long, _ = attention_bias(mask_long)
bias_local_long, _ = attention_bias(mask_long, half_window=w_real)
cos_g_long, sin_g_long = rope_cos_sin(S_long, "full_attention")
cos_l_long, sin_l_long = rope_cos_sin(S_long, "sliding_attention")

h1 = encoder_layer(x_long, 0, bias_global_long, cos_g_long, sin_g_long)
h2 = encoder_layer(h1, 1, bias_local_long, cos_l_long, sin_l_long)
print("x_long:", tuple(x_long.shape), "-> layer 0 (global):", tuple(h1.shape), "-> layer 1 (local):", tuple(h2.shape))

from transformers.masking_utils import create_bidirectional_mask
ref_full_mask = create_bidirectional_mask(config=reference.config, inputs_embeds=x_long, attention_mask=mask_long,
                                          allow_is_bidirectional_skip=False)
cos_gr, sin_gr = reference.rotary_emb(x_long, torch.arange(S_long)[None], "full_attention")
cos_lr, sin_lr = reference.rotary_emb(x_long, torch.arange(S_long)[None], "sliding_attention")
with torch.no_grad():
    ref_h1 = reference.layers[0](x_long, attention_mask=ref_full_mask, position_embeddings=(cos_gr, sin_gr))
    ref_h2 = reference.layers[1](ref_h1, attention_mask=ref_mask, position_embeddings=(cos_lr, sin_lr))

real = mask_long.bool()                                           # compare real tokens only
d1 = (h1[real] - ref_h1[real]).abs().max().item()
d2 = (h2[real] - ref_h2[real]).abs().max().item()
print(f"\nlayer 0 vs reference (real tokens): max |diff| = {d1:.1e}")
print(f"layer 1 vs reference (real tokens): max |diff| = {d2:.1e}")
assert d1 < 1e-4 and d2 < 1e-4

---
# Phase 6 — The full encoder

## Cell 13 — Stack all 28 layers

Embeddings → norm → 28 layers (each with its own mask and RoPE base) → final norm. This is our ModernBERT: every computation is ours except RoPE's cos / sin. **Expect:** 394,781,696 frozen parameters, and a match with the reference `last_hidden_state` to about 3e-5.

In [ ]:
# Cell 13a — Each layer as a module, and the stack

import time

import torch.nn as nn


class ModernBertLayer(nn.Module):
    """Holds one layer's pretrained tensors (shared storage, frozen) and runs encoder_layer() from Cell 11."""

    def __init__(self, layer_idx):
        super().__init__()
        self.layer_idx = layer_idx
        self.is_global = config.is_global_layer(layer_idx)
        prefix = f"layers.{layer_idx}."
        self.params = nn.ParameterDict({
            name[len(prefix):].replace(".", "_"): nn.Parameter(t, requires_grad=False)
            for name, t in weights.items() if name.startswith(prefix)
        })

    def forward(self, h, bias, cos, sin):
        return encoder_layer(h, self.layer_idx, bias, cos, sin)


layers = nn.ModuleList([ModernBertLayer(i) for i in range(config.num_layers)])

### Cell 13b — The forward pass

In [ ]:
# Cell 13b — The forward pass: embeddings → 28 layers → final norm

def modernbert_forward(input_ids, attention_mask, return_all_layers=False):
    """Our ModernBERT encoder: [B, S] token IDs -> [B, S, H] contextual representations."""
    S_ = input_ids.shape[1]
    h = layer_norm(W_emb[input_ids], weights["embeddings.norm.weight"])
    masks = {True: attention_bias(attention_mask)[0],
             False: attention_bias(attention_mask, half_window=config.local_attention_window // 2)[0]}
    ropes = {True: rope_cos_sin(S_, "full_attention"), False: rope_cos_sin(S_, "sliding_attention")}
    all_layers = [h]
    for layer in layers:
        h = layer(h, masks[layer.is_global], *ropes[layer.is_global])
        all_layers.append(h)
    output = layer_norm(h, weights["final_norm.weight"])
    return (output, all_layers) if return_all_layers else output


n_params = sum(p.numel() for p in layers.parameters()) + W_emb.numel() + 2 * H
print(f"layers: {len(layers)} ({sum(l.is_global for l in layers)} global, {sum(not l.is_global for l in layers)} local)")
print(f"parameters (layers + embeddings + 2 norms): {n_params:,} | trainable: {sum(p.numel() for p in layers.parameters() if p.requires_grad)}")
assert n_params == sum(t.numel() for t in weights.values())

### Cell 13c — The whole encoder vs the reference

In [ ]:
# Cell 13c — The whole encoder vs the reference

with torch.no_grad():
    t0 = time.perf_counter()
    ours = modernbert_forward(ids_long, mask_long)
    t_ours = time.perf_counter() - t0
    t0 = time.perf_counter()
    ref_out = reference(input_ids=ids_long, attention_mask=mask_long).last_hidden_state
    t_ref = time.perf_counter() - t0

diff = (ours[real] - ref_out[real]).abs().max().item()
print(f"\ninput_ids {tuple(ids_long.shape)} -> our output {tuple(ours.shape)}")
print(f"ours vs reference last_hidden_state (real tokens): max |diff| = {diff:.1e}")
print(f"forward-pass time on CPU: ours {t_ours:.2f}s | reference {t_ref:.2f}s")
assert diff < 1e-3
encoder_validation_diff = diff

## Cell 14 — Representations layer by layer

Our sentence has `Ġcard` twice: the card whose payment failed, and the new card. At the embedding layer they're the same vector; each layer mixes in context. **Expect:** their similarity drops from 1.0 to about 0.88 (layer 14), ending near 0.94; layer 0 changes the vectors most (≈ 0.59 to the input), later layers refine them (0.92–0.99).

In [ ]:
# Cell 14 — Representations after every layer

with torch.no_grad():
    _, all_hidden = modernbert_forward(input_ids_1, torch.ones_like(input_ids_1), return_all_layers=True)

print(f"captured {len(all_hidden)} hidden states (embeddings + {config.num_layers} layers), each {tuple(all_hidden[0].shape)}")
c1, c2 = 2, 11
assert tokens_1[c1] == tokens_1[c2] == "Ġcard"

cos = torch.nn.functional.cosine_similarity
card_sim = [cos(hs[0, c1], hs[0, c2], dim=0).item() for hs in all_hidden]
step_sim = [float("nan")] + [cos(all_hidden[l][0], all_hidden[l - 1][0], dim=-1).mean().item() for l in range(1, len(all_hidden))]

print(f"\n{'layer':>5} | {'type':>6} | {'cos(card@2, card@11)':>20} | {'mean cos to previous layer':>26}")
for l in range(len(all_hidden)):
    kind = "embed" if l == 0 else ("global" if config.is_global_layer(l - 1) else "local")
    print(f"{l:>5} | {kind:>6} | {card_sim[l]:>20.3f} | {step_sim[l]:>26.3f}")
assert abs(card_sim[0] - 1.0) < 1e-5, "identical at the embedding layer"
assert min(card_sim[1:]) < 0.95, "context separates the two occurrences"
print(f"\nlowest card similarity: {min(card_sim):.3f} at layer {card_sim.index(min(card_sim))}; final layer: {card_sim[-1]:.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(len(all_hidden)), card_sim, marker="o", label="cos(card@2, card@11)")
ax.plot(range(len(all_hidden)), step_sim, marker=".", label="mean cos to previous layer")
ax.set_xlabel("layer (0 = embedding output)")
ax.set_ylabel("cosine similarity")
ax.set_title("The two 'card' tokens diverge as context is mixed in")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

---
# Phase 7 — The output

## Cell 15 — One vector per token, and the masked-LM head

The encoder's output is `[B, S, 1024]`, one contextual vector per token, with no next-token logits. What it's *for* depends on the head on top. The checkpoint ships its pretraining head, masked-LM: fill a `[MASK]` from both sides. **Expect:** for `My card payment [MASK], so I need a new card.`, words like `expired`, `failed`, `bounced`, `declined`.

In [ ]:
# Cell 15 — Encoder output [B, S, H]; a masked-LM demo with the checkpoint's pretraining head

from safetensors import safe_open

masked_text = "My card payment [MASK], so I need a new card."
mlm_batch = tokenizer(masked_text, return_tensors="pt")
mlm_ids = mlm_batch["input_ids"]
with torch.no_grad():
    encoder_out = modernbert_forward(mlm_ids, mlm_batch["attention_mask"])

print("input_ids:", tuple(mlm_ids.shape), "-> encoder output:", tuple(encoder_out.shape), "= [B, S, H]")
assert encoder_out.shape == (1, mlm_ids.shape[1], H), "one vector per token, no vocabulary dimension"

with safe_open(weights_path, "pt") as f:
    head_dense = f.get_tensor("head.dense.weight")                 # [1024, 1024]
    head_norm = f.get_tensor("head.norm.weight")                   # [1024]
    decoder_bias = f.get_tensor("decoder.bias")                    # [V]

mask_pos = (mlm_ids[0] == MASK_ID).nonzero().item()
z = layer_norm(F.gelu(encoder_out[0, mask_pos] @ head_dense.T), head_norm)    # [1024]
logits = z @ W_emb.T + decoder_bias                                            # [V] (tied embeddings)

print(f"\n[MASK] at position {mask_pos}: logits {tuple(logits.shape)}")
probs = torch.softmax(logits, dim=-1)
top = torch.topk(probs, k=5)
mlm_top_prediction = tokenizer.decode([top.indices[0].item()])
print(f"Top predictions for: {masked_text}")
for p_, tid in zip(top.values.tolist(), top.indices.tolist()):
    print(f"  {tokenizer.decode([tid])!r:14s} {p_:.3f}")

## Cell 16 — One vector per sequence: pooling

`[B, S, H]` → `[B, H]`. ModernBERT is pretrained only with masked LM, so `[CLS]` isn't trained to summarize; we use the **mask-aware mean** (also the checkpoint's own `classifier_pooling`). **Expect:** all three methods rank paraphrase > related > unrelated; mean has the clearest margins (0.98 / 0.90 / 0.85). All similarities are high, typical of raw MLM vectors: a trained head reads the small directions that differ (Cell 17).

In [ ]:
# Cell 16 — Sequence representation: [B, S, H] -> [B, H]; we choose mask-aware mean pooling

def mean_pool(hidden, attention_mask):
    m = attention_mask[..., None].to(hidden.dtype)                 # [B, S, 1]
    return (hidden * m).sum(dim=1) / m.sum(dim=1)                  # [B, H]


def cls_pool(hidden, attention_mask):
    return hidden[:, 0]                                            # [B, H]


def max_pool(hidden, attention_mask):
    return hidden.masked_fill(~attention_mask[..., None].bool(), float("-inf")).max(dim=1).values


sentences = ["My card is blocked and I can't use it.",               # anchor
             "I can't use my card because it has been blocked.",    # paraphrase
             "Please send me a replacement debit card.",            # related (card)
             "I want to start a monthly SIP in a mutual fund."]     # unrelated
batch4 = tokenizer(sentences, padding=True, return_tensors="pt")
with torch.no_grad():
    out4 = modernbert_forward(batch4["input_ids"], batch4["attention_mask"])
print("encoder output:", tuple(out4.shape), "-> pooled:", tuple(mean_pool(out4, batch4["attention_mask"]).shape))

print(f"\n{'pooling':>7} | {'paraphrase':>10} | {'related':>8} | {'unrelated':>9} | ranks paraphrase > related > unrelated?")
for name, pool in [("[CLS]", cls_pool), ("mean", mean_pool), ("max", max_pool)]:
    v = pool(out4, batch4["attention_mask"])
    sims = torch.nn.functional.cosine_similarity(v[0:1], v[1:], dim=-1)
    print(f"{name:>7} | {sims[0]:10.3f} | {sims[1]:8.3f} | {sims[2]:9.3f} | {bool(sims[0] > sims[1] > sims[2])}")

print(f"\ncheckpoint's own classifier_pooling: {raw_config['classifier_pooling']!r}")
print("Chosen: mask-aware MEAN pooling.")
assert raw_config["classifier_pooling"] == "mean"

---
# Phase 8 — A classification head on the encoder

## Cell 17 — Train a fixed head

The key idea: the frozen encoder does the language work, and a small **head** reads a decision off the pooled vector. Here: `Linear(1024, 6)` for 6 customer-message intents, trained on 8 labelled messages each, then tested on 18 unseen ones. Only the head's 6,150 numbers learn. The vectors are standardized first (raw MLM vectors share a common direction, Cell 16). This is how "BERT for classification" works, and it suits **fixed** label sets: sentiment, spam, a fixed intent list.

In [ ]:
# Cell 17a — A small labelled dataset, and its frozen sentence vectors

TRAIN = {
    "Cards": ["I lost my debit card yesterday.", "My credit card has been stolen, please block it.",
              "The card was swallowed by the ATM.", "Please freeze my card, I can't find it.",
              "My card is blocked after three wrong PINs.", "I need a replacement card, mine is damaged.",
              "Someone used my card details online.", "How do I unblock my card?"],
    "Transfers": ["My bank transfer failed this morning.", "The money I sent hasn't arrived after three days.",
                  "Why is my payment still pending?", "I sent money to the wrong account number.",
                  "A transfer was rejected but the money left my account.", "How long does an international transfer take?",
                  "I was charged twice for one payment.", "Please cancel the transfer I made an hour ago."],
    "Investments": ["I want to start a monthly SIP.", "How are my mutual funds performing?",
                    "Can I redeem part of my investment?", "Which equity fund do you recommend for ten years?",
                    "Please switch my portfolio to lower risk.", "What's the exit load on this fund?",
                    "I'd like to invest a lump sum in bonds.", "Show me the returns on my portfolio this year."],
    "Insurance": ["I want to file a claim for my car accident.", "When is my health insurance premium due?",
                  "My policy expired last week, can I renew it?", "Does my travel insurance cover lost luggage?",
                  "Please add my wife to my health policy.", "How much is the premium for life cover?",
                  "My claim was rejected, why?", "I need a copy of my insurance policy."],
    "Loans": ["I'd like to apply for a personal loan.", "What interest rate do you offer on home loans?",
              "Can I prepay my car loan without a penalty?", "My loan EMI was debited twice.",
              "How much can I borrow on my salary?", "Please send my loan statement.",
              "I want to extend my loan tenure.", "Is my loan application approved yet?"],
    "Profile": ["I moved house, please update my address.", "How do I change the phone number on my account?",
                "My email address is wrong in your records.", "Please correct the spelling of my name.",
                "I want to add a nominee to my account.", "Update my mailing address to the new office.",
                "I got married and need to change my surname.", "Can I change my registered email online?"],
}
TEST = {
    "Cards": ["My card got stuck in the machine and didn't come back.", "Block my card please, my wallet was stolen.",
              "Can you send me a new card? This one is cracked."],
    "Transfers": ["I paid my landlord but he says he got nothing.", "The payment shows as processing since Monday.",
                  "I transferred to a closed account, where's my money?"],
    "Investments": ["Is it a good time to add to my index fund?", "Please stop my monthly SIP from next month.",
                    "How much have my funds gained since January?"],
    "Insurance": ["My phone was stolen on holiday, will the policy pay?", "Can I increase the sum insured on my health plan?",
                  "The hospital bill claim is still not settled."],
    "Loans": ["I need money for a wedding, what loan can I get?", "How do I close my home loan early?",
              "What will my monthly EMI be for 5 lakh?"],
    "Profile": ["Please change my contact number to the new one.", "My address on the statement is the old one.",
                "How can I update my email for alerts?"],
}
CLASSES = list(TRAIN)
assert list(TEST) == CLASSES and all(len(v) == 8 for v in TRAIN.values()) and all(len(v) == 3 for v in TEST.values())


def encode(texts):
    """The frozen encoder (Cell 13) + mean pooling (Cell 16): N messages -> [N, H]."""
    batch = tokenizer(texts, padding=True, return_tensors="pt")
    with torch.no_grad():
        return mean_pool(modernbert_forward(batch["input_ids"], batch["attention_mask"]), batch["attention_mask"])


def dataset(by_class, classes):
    texts = [t for c in classes for t in by_class[c]]
    labels = torch.tensor([k for k, c in enumerate(classes) for _ in by_class[c]])
    return texts, labels


train_texts, y_train = dataset(TRAIN, CLASSES)
test_texts, y_test = dataset(TEST, CLASSES)
v_train, v_test = encode(train_texts), encode(test_texts)
mu, sd = v_train.mean(dim=0), v_train.std(dim=0) + 1e-6               # standardize with the TRAINING statistics


def standardize(v):
    return (v - mu) / sd


X_train, X_test = standardize(v_train), standardize(v_test)
print(f"{len(CLASSES)} classes | train {tuple(X_train.shape)} | test {tuple(X_test.shape)}")

### Cell 17b — Train the head

Cross-entropy, Adam, 300 full-batch steps. `train_head` is reused in Cell 18c.

In [ ]:
# Cell 17b — Train a Linear(H, 6) head on the frozen vectors

def train_head(X, y, n_classes, steps=300, seed=42):
    torch.manual_seed(seed)
    head = nn.Linear(X.shape[1], n_classes)
    opt = torch.optim.Adam(head.parameters(), lr=1e-2, weight_decay=1e-3)
    for step in range(steps + 1):
        loss = F.cross_entropy(head(X), y)
        if step % 100 == 0:
            acc = (head(X).argmax(dim=1) == y).float().mean().item()
            print(f"  step {step:3d} | loss {loss.item():.4f} | train accuracy {acc:.2f}")
        opt.zero_grad()
        loss.backward()
        opt.step()
    return head


fixed_head = train_head(X_train, y_train, len(CLASSES))
print("W:", tuple(fixed_head.weight.shape), "| b:", tuple(fixed_head.bias.shape),
      f"| trained parameters: {sum(p.numel() for p in fixed_head.parameters()):,} (the encoder's {n_params:,} stay frozen)")

with torch.no_grad():
    logits = fixed_head(X_test)                                        # [18, 6]
    manual = X_test[0] @ fixed_head.weight[2] + fixed_head.bias[2]
assert torch.allclose(manual, logits[0, 2], atol=1e-4)
print("logit k = w_k · z + b_k  (checked for test message 0, class 2)")

### Cell 17c — Test on unseen messages

In [ ]:
# Cell 17c — Accuracy on the 18 unseen test messages

def report(head, X, y, texts, classes):
    with torch.no_grad():
        probs = torch.softmax(head(X), dim=1)
    pred = probs.argmax(dim=1)
    for t, yt, p, pr in zip(texts, y.tolist(), pred.tolist(), probs.max(dim=1).values.tolist()):
        mark = "" if p == yt else f"   <- wrong (truth: {classes[yt]})"
        print(f"  {t[:58]:58s} -> {classes[p]:12s} {pr:.2f}{mark}")
    acc = (pred == y).sum().item()
    print(f"accuracy: {acc} / {len(y)}")
    return acc


test_correct_6 = report(fixed_head, X_test, y_test, test_texts, CLASSES)

## Cell 18 — What a fixed head can't do

Each class is just a learned row of `W`: the head never reads what a class *means*, and a new class needs **labelled examples and retraining**. Fine for fixed labels; the wrong shape for agents that are added and re-scoped at run time.

| | Fixed head (here) | Compatibility head (02 onward) |
|---|---|---|
| Input | message | message **+ candidate description** |
| Output | one logit per known class | one score per (message, candidate) |
| New class | new row + labelled examples + retraining | write a description |

In [ ]:
# Cell 18a — The head never reads what a class means

message = "My bank transfer failed."
descriptions_v1 = {"Transfers": "Handles bank transfers and payment failures.", "Cards": "Handles card blocking."}
descriptions_v2 = {"Transfers": "Handles ONLY mortgage questions now.", "Cards": "Handles bank transfers too."}

z = standardize(encode([message]))
with torch.no_grad():
    logits_v1 = fixed_head(z)          # descriptions_v1 are not an input...
    logits_v2 = fixed_head(z)          # ...and neither are descriptions_v2
print("prediction with descriptions v1:", CLASSES[logits_v1.argmax().item()], "| with v2:", CLASSES[logits_v2.argmax().item()])
assert torch.equal(logits_v1, logits_v2)
print("-> identical: rewriting what the classes mean changes nothing, because the head never reads descriptions.")

### Cell 18b — Add a 7th class, untrained

"App & login" as a new random row: its test messages go to the old classes.

In [ ]:
# Cell 18b — A 7th class as an untrained row

NEW = "App & login"
TRAIN[NEW] = ["I can't log in to the mobile app.", "The app crashes every time I open it.",
              "I'm not receiving the OTP to sign in.", "My internet banking password expired.",
              "The app says my device is not registered.", "I forgot my login PIN for the app.",
              "Face ID login stopped working after the update.", "My online banking account is locked."]
TEST[NEW] = ["The app keeps logging me out.", "I never get the verification code by SMS.", "How do I reset my net banking password?"]
CLASSES_7 = CLASSES + [NEW]

torch.manual_seed(0)
head_7 = nn.Linear(H, 7)
with torch.no_grad():
    head_7.weight[:6], head_7.bias[:6] = fixed_head.weight, fixed_head.bias   # the 6 trained rows; row 6 is random

new_texts, y_new = TEST[NEW], torch.full((3,), 6)
X_new = standardize(encode(new_texts))
print(f"weight {tuple(fixed_head.weight.shape)} -> {tuple(head_7.weight.shape)}; row 6 is untrained\n")
print(f"The new class's test messages, untrained row:")
new_correct_before = report(head_7, X_new, y_new, new_texts, CLASSES_7)

### Cell 18c — Retrain with 8 examples of it

The new class is recognized, and the old ones still work.

In [ ]:
# Cell 18c — Retrain on 7 classes, then test old and new

train_texts_7, y_train_7 = dataset(TRAIN, CLASSES_7)
test_texts_7, y_test_7 = dataset(TEST, CLASSES_7)
X_train_7 = standardize(encode(train_texts_7))            # the same standardization as before
X_test_7 = standardize(encode(test_texts_7))

head_7 = train_head(X_train_7, y_train_7, 7)
print()
test_correct_7 = report(head_7, X_test_7, y_test_7, test_texts_7, CLASSES_7)
new_correct_after = sum(p == 6 for p in head_7(X_test_7[-3:]).argmax(dim=1).tolist())
print(f"\nnew class: {new_correct_before} / 3 before retraining -> {new_correct_after} / 3 after; "
      f"the price was 8 labelled examples and a new training run.")

---
# Next

A head on a pooled encoder vector works for fixed labels, but every new class costs data and retraining. **`02_frozen_router.ipynb`** (branch `frozen-router`) keeps the encoder frozen and changes the head: the agent's description goes into the input, and the head scores each (conversation, agent) pair, so a new agent needs no retraining. **`03_lora_signpost.ipynb`** (branch `signpost`) then fine-tunes the encoder itself (LoRA), which became the published model, **Signpost**.